In [1]:
import os
import math
import json
import torch
import random
import torchaudio
import torchaudio.transforms as T
import numpy as np
from tqdm import tqdm
from torch import nn, optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
from sklearn.metrics import *
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
RESNET_CKPT_PATH="/kaggle/input/models/rudranshiverma/deepfake-resnet18/pytorch/default/1/resnet18-model.pth"
 
checkpoint = torch.load(RESNET_CKPT_PATH, map_location="cpu", weights_only=False)
 
mel_mean=checkpoint["mel_mean"]
mel_std=checkpoint["mel_std"]
delta_mean=checkpoint["delta_mean"]
delta_std=checkpoint["delta_std"]
delta2_mean=checkpoint["delta2_mean"]
delta2_std=checkpoint["delta2_std"]
ORIGINAL_EER=0.0662   
ORIGINAL_PR_AUC=checkpoint["pr_auc"]
print(f"mel mean={mel_mean:.4f} std={mel_std:.4f}")
print(f"delta mean={delta_mean:.4f} std={delta_std:.4f}")
print(f"delta2 mean={delta2_mean:.4f} std={delta2_std:.4f}")
print(f"Checkpoint epoch: {checkpoint['epoch']} PR-AUC: {checkpoint['pr_auc']:.4f}")

mel mean=-38.0290 std=38.5913
delta mean=-0.1480 std=3.9119
delta2 mean=-0.0000 std=1.6467
Checkpoint epoch: 6 PR-AUC: 1.0000


In [3]:
TARGET_SR = 16_000
MAX_LEN= TARGET_SR * 4

In [4]:
def rawboost_augment(waveform: torch.Tensor) -> torch.Tensor:
    #Additive Gaussian noise at a random SNR between 15-40 dB.
    snr_db=random.uniform(15, 40)
    sig_pow=waveform.pow(2).mean()
    noise=torch.randn_like(waveform)
    noise_pow=noise.pow(2).mean()
    scale=torch.sqrt(sig_pow/(noise_pow * 10 ** (snr_db / 10) + 1e-8))
    return waveform + scale * noise
def add_real_noise(waveform, noise_files):
    noise_path = random.choice(noise_files)
    noise, sr = torchaudio.load(noise_path)

    if noise.shape[0] > 1:
        noise = noise.mean(dim=0, keepdim=True)

    if sr != 16000:
        noise = T.Resample(sr, 16000)(noise)
    if noise.shape[1] < waveform.shape[1]:
        repeat = waveform.shape[1] // noise.shape[1] + 1
        noise = noise.repeat(1, repeat)

    noise = noise[:, :waveform.shape[1]]

    # SNR
    snr_db = random.uniform(10, 30)
    sig_pow = waveform.pow(2).mean()
    noise_pow = noise.pow(2).mean()
    scale = torch.sqrt(sig_pow / (noise_pow * 10 ** (snr_db / 10) + 1e-8))

    return waveform + scale * noise

def apply_waveform_augment(waveform: torch.Tensor, noise_files: list) -> torch.Tensor:
    #at most one waveform augmentation per sample.
    r = random.random()
    if r < 0.33:
        return rawboost_augment(waveform)
    elif r < 0.55 and noise_files:
        return add_real_noise(waveform, noise_files)
    return waveform   

In [7]:
class UnifiedDataset(Dataset):
    def __init__(
        self,
        protocol_path: str,
        flac_dir: str,
        extra_files: list = None,
        extra_labels: list = None,
        extra_sources: list = None,
        augment: bool = False,
        noise_files: list = None,
    ):
        self.augment=augment
        self.noise_files = noise_files if noise_files else []
        self.samples=[]
 
        with open(protocol_path) as f:
            for line in f:
                parts = line.strip().split()
                fname = parts[1]
                label = 0 if parts[4] == "bonafide" else 1
                self.samples.append({
                    "path":os.path.join(flac_dir, fname + ".flac"),
                    "label":label,
                    "source":"asv",
                })
 
        if extra_files:
            for i, (path, label) in enumerate(zip(extra_files, extra_labels)):
                src = extra_sources[i] if extra_sources else "extra"
                self.samples.append({
                    "path": path, "label": label, "source": src,
                })
 
    def __len__(self):
        return len(self.samples)
 
    def _process_audio(self, path: str) -> torch.Tensor:
        mel_tf = T.MelSpectrogram(
            sample_rate=TARGET_SR, n_fft=1024, hop_length=256, n_mels=128
        )
        a2db = T.AmplitudeToDB()
 
        waveform, sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != TARGET_SR:
            waveform = T.Resample(sr, TARGET_SR)(waveform)
        if waveform.shape[1] > MAX_LEN:
            waveform = waveform[:, :MAX_LEN]
        else:
            waveform = torch.nn.functional.pad(
                waveform, (0, MAX_LEN - waveform.shape[1])
            )
 
        if self.augment:
            waveform = apply_waveform_augment(waveform, self.noise_files)
 
        mel=mel_tf(waveform)
        mel=a2db(mel)
        delta=torchaudio.functional.compute_deltas(mel)
        delta2=torchaudio.functional.compute_deltas(delta)
 
        mel=(mel- mel_mean)/(mel_std+ 1e-6)
        delta= (delta- delta_mean)/ (delta_std  + 1e-6)
        delta2 = (delta2 - delta2_mean) / (delta2_std + 1e-6)
 
        return torch.cat([mel, delta, delta2], dim=0)  # [3, 128, T]
 
    def _spec_augment(self, mel: torch.Tensor) -> torch.Tensor:
        C, F, T_len = mel.shape
        mel = mel.clone()
        f_mask = random.randint(0, 20)
        f0= random.randint(0, max(1, F - f_mask - 1))
        mel[0, f0:f0 + f_mask, :] = 0
        t_mask = random.randint(0, 40)
        t0== random.randint(0, max(1, T_len - t_mask - 1))
        mel[:, :, t0:t0 + t_mask] = 0
        return mel
 
    def __getitem__(self, idx):
        item = self.samples[idx]
        mel= self._process_audio(item["path"])
        if self.augment:
            mel = self._spec_augment(mel)
        return mel.float(), torch.tensor(item["label"], dtype=torch.float32)

In [8]:
train_protocol = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
dev_protocol   = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
eval_protocol  = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"
 
train_flac_dir = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_train/flac"
dev_flac_dir   = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_dev/flac"
eval_flac_dir  = "/kaggle/input/datasets/rudranshiverma/asv-spoof-la-track/data/ASVspoof2019_LA_eval/flac"

In [9]:
tts_dir= "/kaggle/input/datasets/rudranshiverma/modern-tts-dataset/Fake_ElevenLabs_Respeecher"
tts_files  = [os.path.join(tts_dir, f) for f in os.listdir(tts_dir) if f.endswith(".wav")]
tts_labels = [1] * len(tts_files)
print(f"TTS files found: {len(tts_files)}")

libri_dir= "/kaggle/input/datasets/rudranshiverma/librispeech-subset"
libri_files= [os.path.join(libri_dir, f) for f in os.listdir(libri_dir) if f.endswith(".wav")]
libri_labels = [0] * len(libri_files)
print(f"LibriSpeech files found: {len(libri_files)}")

noise_dir = "/kaggle/input/datasets/rudranshiverma/noise-dataset/audio"
noise_files = [
    os.path.join(noise_dir, f)
    for f in os.listdir(noise_dir)
    if f.endswith(".wav")
]
print("Noise files:", len(noise_files))

TTS files found: 600
LibriSpeech files found: 1000
Noise files: 2000


In [10]:
random.seed(42)
extra_all = list(zip(
    tts_files+ libri_files,
    tts_labels+ libri_labels,
    ["tts"] * len(tts_files) + ["libri"] * len(libri_files),
))
random.shuffle(extra_all)
 
split= int(0.8 * len(extra_all))
extra_train= extra_all[:split]
extra_val= extra_all[split:]
 
if extra_train:
    extra_train_files, extra_train_labels, extra_train_sources = zip(*extra_train)
    extra_train_files= list(extra_train_files)
    extra_train_labels= list(extra_train_labels)
    extra_train_sources = list(extra_train_sources)
else:
    extra_train_files = extra_train_labels = extra_train_sources = []
 
if extra_val:
    extra_val_files, extra_val_labels, extra_val_sources = zip(*extra_val)
    extra_val_files= list(extra_val_files)
    extra_val_labels= list(extra_val_labels)
    extra_val_sources = list(extra_val_sources)
else:
    extra_val_files = extra_val_labels = extra_val_sources = []
 
n_tts_train= extra_train_sources.count("tts")
n_libri_train = extra_train_sources.count("libri")
print(f"Extra train: {len(extra_train_files)}  tts={n_tts_train}  libri={n_libri_train}")
print(f"Extra val:{len(extra_val_files)}")
 
 

Extra train: 1280  tts=475  libri=805
Extra val:   320


In [11]:
train_dataset = UnifiedDataset(
    protocol_path= train_protocol,
    flac_dir= train_flac_dir,
    extra_files= extra_train_files,
    extra_labels= extra_train_labels,
    extra_sources= extra_train_sources,
    noise_files= noise_files,
    augment= True,
)
dev_dataset = UnifiedDataset(
    protocol_path = dev_protocol,
    flac_dir= dev_flac_dir,
    augment= False,
)
# Extra-only val set
extra_val_dataset = UnifiedDataset(
    protocol_path = dev_protocol,
    flac_dir= dev_flac_dir,
    extra_files= extra_val_files,
    extra_labels= extra_val_labels,
    extra_sources = extra_val_sources,
    augment= False,
)
extra_val_dataset.samples = [s for s in extra_val_dataset.samples if s["source"] != "asv"]
 
n_asv_bon= sum(1 for s in train_dataset.samples if s["source"]=="asv" and s["label"]==0)
n_asv_spo= sum(1 for s in train_dataset.samples if s["source"]=="asv" and s["label"]==1)
n_tts_spo= sum(1 for s in train_dataset.samples if s["source"]=="tts")
n_lib_bon= sum(1 for s in train_dataset.samples if s["source"]=="libri")
 
print(f"\nTrain set:")
print(f"  ASV bonafide : {n_asv_bon}")
print(f"  ASV spoof    : {n_asv_spo}")
print(f"  TTS spoof    : {n_tts_spo}")
print(f"  Libri bonafide: {n_lib_bon}")
print(f"  Total        : {len(train_dataset.samples)}")
print(f"\nDev (ASVspoof): {len(dev_dataset.samples)}")
print(f"Extra val:      {len(extra_val_dataset.samples)}")
 


Train set:
  ASV bonafide : 2580
  ASV spoof    : 22800
  TTS spoof    : 475
  Libri bonafide: 805
  Total        : 26660

Dev (ASVspoof): 24844
Extra val:      320


In [12]:
TARGET_ASV_BON = 1/3
TARGET_ASV_SPO = 1/3
TARGET_TTS_SPO = 1/6
TARGET_LIB_BON = 1/6
 
w_asv_bon = TARGET_ASV_BON / max(n_asv_bon, 1)
w_asv_spo = TARGET_ASV_SPO / max(n_asv_spo, 1)
w_tts_spo = TARGET_TTS_SPO / max(n_tts_spo, 1)
w_lib_bon = TARGET_LIB_BON / max(n_lib_bon, 1)
 
print(f"Per-sample weights:")
print(f"  ASV bonafide:  {w_asv_bon:.8f}")
print(f"  ASV spoof:     {w_asv_spo:.8f}")
print(f"  TTS spoof:     {w_tts_spo:.8f}")
print(f"  Libri bonafide:{w_lib_bon:.8f}")
 
def get_sample_weight(sample: dict) -> float:
    src, lbl = sample["source"], sample["label"]
    if src == "asv" and lbl == 0: return w_asv_bon
    if src == "asv" and lbl == 1: return w_asv_spo
    if src == "tts":               return w_tts_spo
    if src == "libri":             return w_lib_bon
    # fallback is class weight
    return (1/3) / max(sum(1 for s in train_dataset.samples if s["label"]==lbl), 1)
 
sample_weights = [get_sample_weight(s) for s in train_dataset.samples]
total_w = sum(sample_weights)
bs = 32
print(f"\nExpected batch composition (batch_size={bs}):")
print(f"  ASV bonafide:   {bs * n_asv_bon*w_asv_bon / total_w:.1f}")
print(f"  ASV spoof:      {bs * n_asv_spo*w_asv_spo / total_w:.1f}")
print(f"  TTS spoof:      {bs * n_tts_spo*w_tts_spo / total_w:.1f}")
print(f"  Libri bonafide: {bs * n_lib_bon*w_lib_bon / total_w:.1f}")
 
sampler = WeightedRandomSampler(
    weights= sample_weights,
    num_samples = len(sample_weights),
    replacement = True,
)
 
train_loader = DataLoader(
    train_dataset, batch_size=32, sampler=sampler,
    num_workers=2, pin_memory=True,
)
dev_loader = DataLoader(
    dev_dataset, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True,
)
extra_val_loader = DataLoader(
    extra_val_dataset, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True,
)
print(f"\nTrain batches: {len(train_loader)}")
print(f"Dev batches:{len(dev_loader)}")
print(f"Extra val batches: {len(extra_val_loader)}")
 
 

Per-sample weights:
  ASV bonafide:  0.00012920
  ASV spoof:     0.00001462
  TTS spoof:     0.00035088
  Libri bonafide:0.00020704

Expected batch composition (batch_size=32):
  ASV bonafide:   10.7
  ASV spoof:      10.7
  TTS spoof:      5.3
  Libri bonafide: 5.3

Train batches: 834
Dev batches:   389
Extra val batches: 5


In [13]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
 
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )
        pt = torch.exp(-bce)
        at = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (at * (1 - pt) ** self.gamma * bce).mean()
 
 
def build_model_original_arch(device):
    """Exact architecture from original training — must match for load_state_dict."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for name, param in model.named_parameters():
        if "layer2" not in name and "layer3" not in name and "layer4" not in name:
            param.requires_grad = False
    in_feat  = model.fc.in_features
    model.fc = nn.Sequential(
        nn.BatchNorm1d(in_feat),
        nn.Dropout(0.3),
        nn.Linear(in_feat, 1),
    )
    return model.to(device)
 
 
model = build_model_original_arch(device)
model.load_state_dict(checkpoint["model_state_dict"])
print("Checkpoint loaded successfully.")
 
#finetuned only layer4 + fc — layer2/3 keep their ASVspoof features
for name, param in model.named_parameters():
    param.requires_grad = ("layer4" in name or "fc" in name)
 
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total= sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}({100*trainable/total:.1f}%)")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 156MB/s] 


Checkpoint loaded successfully.
Trainable params: 8,395,265 / 11,178,049  (75.1%)


In [14]:
EPOCHS= 20
MIN_EPOCHS= 5
PATIENCE= 5
LR= 5e-5
MIN_LR= 1e-6
 
# DEV EER regression guard
DEV_EER_GUARD= ORIGINAL_EER * 1.5  
EXTRA_RECALL_GUARD = 0.88
 
criterion = FocalLoss(alpha=0.75, gamma=2.0)
optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-4,
)
 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)
 
print(f"DEV EER guard:        <= {DEV_EER_GUARD:.4f} (1.5x original {ORIGINAL_EER:.4f})")
print(f"Extra recall guard:   >= {EXTRA_RECALL_GUARD}")
print(f"Save criterion:       extra_val PR-AUC improves (not saturated)")

DEV EER guard:        <= 0.0993 (1.5x original 0.0662)
Extra recall guard:   >= 0.88
Save criterion:       extra_val PR-AUC improves (not saturated)


In [16]:
def evaluate(model, loader, loader_name=""):
    model.eval()
    all_scores, all_labels = [], []
    val_loss = 0.0
 
    with torch.no_grad():
        for inputs, labels in loader:
            inputs= inputs.to(device)
            labels_d  = labels.to(device).unsqueeze(1)
            outputs= model(inputs)
            val_loss += criterion(outputs, labels_d).item()
            scores= torch.sigmoid(outputs).view(-1)
            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(labels.numpy())
 
    scores_np = np.array(all_scores)
    labels_np = np.array(all_labels)
    avg_loss  = val_loss / len(loader)
 
    if len(np.unique(labels_np)) < 2:
        print(f"  [{loader_name}] Only one class — skipping AUC metrics.")
        return {"val_loss": avg_loss}
 
    pr_auc  = average_precision_score(labels_np, scores_np)
    roc_auc = roc_auc_score(labels_np, scores_np)
 
    fpr, tpr, thresholds_roc = roc_curve(labels_np, scores_np)
    fnr= 1 - tpr
    eer_idx= np.nanargmin(np.abs(fnr - fpr))
    eer= fpr[eer_idx]
    eer_thresh = thresholds_roc[eer_idx]
 
    prec_arr, rec_arr, thresholds_pr = precision_recall_curve(labels_np, scores_np)
 
    # Threshold: highest precision where recall >= 0.90.
    # Fallback: max-F1
    valid = np.where(rec_arr[:-1] >= 0.90)[0]
    if len(valid) > 0:
        best_idx         = valid[np.argmax(prec_arr[valid])]
        recall_threshold = thresholds_pr[best_idx]
        fallback_used    = False
    else:
        f1s= (2 * prec_arr[:-1] * rec_arr[:-1] /(prec_arr[:-1] + rec_arr[:-1] + 1e-8))
        best_idx= np.argmax(f1s)
        recall_threshold = thresholds_pr[best_idx]
        fallback_used= True
        print(f"  [{loader_name}] WARNING: recall>=0.90 not achievable. "
              f"Fallback max-F1 threshold={recall_threshold:.4f}")
 
    preds= (scores_np > recall_threshold).astype(int)
    accuracy  = accuracy_score(labels_np, preds)
    precision = precision_score(labels_np, preds, zero_division=0)
    recall= recall_score(labels_np, preds, zero_division=0)
    f1= f1_score(labels_np, preds, zero_division=0)
    cm= confusion_matrix(labels_np, preds)
    missed= int(cm[1][0]) if cm.shape == (2, 2) else -1
 
    print(f"  [{loader_name}] "
          f"PR-AUC={pr_auc:.4f}  ROC-AUC={roc_auc:.4f}  EER={eer:.4f}  "
          f"Recall={recall:.4f}  Precision={precision:.4f}  "
          f"F1={f1:.4f}  Missed={missed}")
    print(f"  [{loader_name}] "
          f"score min={scores_np.min():.3f}  max={scores_np.max():.3f}  "
          f"mean={scores_np.mean():.3f}  std={scores_np.std():.3f}")
    print(f"  [{loader_name}] Confusion matrix:\n{cm}")
 
    return {
        "val_loss":float(avg_loss),
        "pr_auc":float(pr_auc),
        "roc_auc":float(roc_auc),
        "eer":float(eer),
        "eer_threshold":float(eer_thresh),
        "recall_threshold": float(recall_threshold),
        "accuracy":float(accuracy),
        "precision":float(precision),
        "recall":float(recall),
        "f1":float(f1),
        "missed":missed,
        "fallback_used":fallback_used,
        "confusion_matrix": cm.tolist(),
    }
 
 

In [17]:
best_hmean = -1.0
best_dev_eer      = float("inf")
epochs_no_improve = 0
history           = {
    "train_loss":[],
    "dev_val_loss":[], "dev_pr_auc":  [], "dev_roc_auc": [],
    "dev_eer":[], "dev_recall":  [], "dev_precision":[],
    "dev_f1":[], "dev_missed":  [],
    "extra_pr_auc":[], "extra_eer":[], "extra_recall": [],
}
 
print(f"\nFinetuning for up to {EPOCHS} epochs")
print(f"Primary save criterion: extra_val PR-AUC improves (new-domain metric, not saturated)")
print(f"Guard 1:  dev EER <= {DEV_EER_GUARD:.4f}")
print(f"Guard 2:  extra_val recall >= {EXTRA_RECALL_GUARD}")
print(f"Early stopping: patience={PATIENCE} after min_epochs={MIN_EPOCHS}")
print(f"Baseline: original model EER={ORIGINAL_EER:.4f}  PR-AUC={ORIGINAL_PR_AUC:.4f}\n")
 
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        inputs = inputs.to(device)
        labels = labels.unsqueeze(1).to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
 
    avg_train_loss = running_loss / len(train_loader)
 
    #evaluate
    print(f"\nEpoch {epoch+1}/{EPOCHS}  "
          f"train_loss={avg_train_loss:.6f}  "
          f"lr={optimizer.param_groups[0]['lr']:.2e}")
 
    dev_m   = evaluate(model, dev_loader,       "ASVspoof-dev")
    extra_m = evaluate(model, extra_val_loader, "Extra-val")
 
    scheduler.step(dev_m["val_loss"])
 
    history["train_loss"].append(avg_train_loss)
    history["dev_val_loss"].append(dev_m.get("val_loss", 0))
    history["dev_pr_auc"].append(dev_m.get("pr_auc", 0))
    history["dev_roc_auc"].append(dev_m.get("roc_auc", 0))
    history["dev_eer"].append(dev_m.get("eer", 1))
    history["dev_recall"].append(dev_m.get("recall", 0))
    history["dev_precision"].append(dev_m.get("precision", 0))
    history["dev_f1"].append(dev_m.get("f1", 0))
    history["dev_missed"].append(dev_m.get("missed", -1))
    history["extra_pr_auc"].append(extra_m.get("pr_auc", 0))
    history["extra_eer"].append(extra_m.get("eer", 1))
    history["extra_recall"].append(extra_m.get("recall", 0))
 
    dev_eer= dev_m.get("eer", 1.0)
    dev_recall= dev_m.get("recall", 0.0)
    extra_pr_auc = extra_m.get("pr_auc", 0.0)
    extra_recall = extra_m.get("recall", 0.0)


    eer_ok= dev_eer<= DEV_EER_GUARD      
    dev_recall_ok  = dev_recall   >= 0.90               
    extra_recall_ok= extra_recall >= EXTRA_RECALL_GUARD  

    hmean = (2 * extra_pr_auc * dev_recall) / (extra_pr_auc + dev_recall + 1e-9)

    all_guards_ok = eer_ok and dev_recall_ok and extra_recall_ok

    if all_guards_ok and hmean > best_hmean:
        best_hmean= hmean
        best_dev_eer= min(best_dev_eer, dev_eer)
        epochs_no_improve = 0

        torch.save({
        "epoch":epoch + 1,
        "model_state_dict":model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "pr_auc":dev_m.get("pr_auc", 0),
        "roc_auc":dev_m.get("roc_auc", 0),
        "eer":dev_eer,
        "eer_threshold":dev_m.get("eer_threshold", 0.5),
        "recall_threshold":dev_m.get("recall_threshold", 0.5),
        "recall":dev_recall,
        "precision":dev_m.get("precision", 0),
        "accuracy":dev_m.get("accuracy", 0),
        "f1":dev_m.get("f1", 0),
        "confusion_matrix":dev_m.get("confusion_matrix", []),
        "extra_val_pr_auc":extra_pr_auc,
        "extra_val_eer":extra_m.get("eer", 1),
        "extra_val_recall":extra_recall,
        "save_hmean":hmean,
        "mel_mean":mel_mean,"mel_std":mel_std,
        "delta_mean":  delta_mean,"delta_std":delta_std,
        "delta2_mean": delta2_mean, "delta2_std":  delta2_std,
        "finetune_dataset": "asvspoof2019+elevenlabs+librispeech+noise",
    }, "/kaggle/working/resnet18_finetuned.pth")

        print(f"  ✓ SAVED  hmean={hmean:.4f}  "
          f"dev_recall={dev_recall:.4f}  extra_PR={extra_pr_auc:.4f}  "
          f"dev_EER={dev_eer:.4f}  dev_Missed={dev_m.get('missed',-1)}")

    else:
        epochs_no_improve += 1
        reasons = []
        if not eer_ok:reasons.append(f"dev_EER={dev_eer:.4f}>{DEV_EER_GUARD:.4f}")
        if not dev_recall_ok:reasons.append(f"dev_recall={dev_recall:.4f}<0.90")
        if not extra_recall_ok: reasons.append(f"extra_recall={extra_recall:.4f}<{EXTRA_RECALL_GUARD}")
        if all_guards_ok:reasons.append(f"hmean={hmean:.4f}<=best={best_hmean:.4f}")
        print(f"  No improvement {epochs_no_improve}/{PATIENCE}  ({' | '.join(reasons)})")

    if epochs_no_improve >= PATIENCE and (epoch + 1) >= MIN_EPOCHS:
        print(f"\nEarly stopping at epoch {epoch+1}.  "
          f"Best hmean={best_hmean:.4f}  Best dev EER={best_dev_eer:.4f}")
        break       


Finetuning for up to 20 epochs
Primary save criterion: extra_val PR-AUC improves (new-domain metric, not saturated)
Guard 1:  dev EER <= 0.0993
Guard 2:  extra_val recall >= 0.88
Early stopping: patience=5 after min_epochs=5
Baseline: original model EER=0.0662  PR-AUC=1.0000



Epoch 1/20: 100%|██████████| 834/834 [04:52<00:00,  2.86it/s]


Epoch 1/20  train_loss=0.027009  lr=5.00e-05


  [ASVspoof-dev] PR-AUC=1.0000  ROC-AUC=0.9998  EER=0.0067  Recall=0.9132  Precision=1.0000  F1=0.9546  Missed=1936
  [ASVspoof-dev] score min=0.001  max=1.000  mean=0.875  std=0.274
  [ASVspoof-dev] Confusion matrix:
[[ 2548     0]
 [ 1936 20360]]
  [Extra-val] PR-AUC=0.9935  ROC-AUC=0.9953  EER=0.0410  Recall=0.9120  Precision=0.9913  F1=0.9500  Missed=11
  [Extra-val] score min=0.047  max=0.999  mean=0.461  std=0.366
  [Extra-val] Confusion matrix:
[[194   1]
 [ 11 114]]
  ✓ SAVED  hmean=0.9517  dev_recall=0.9132  extra_PR=0.9935  dev_EER=0.0067  dev_Missed=1936


Epoch 2/20: 100%|██████████| 834/834 [04:27<00:00,  3.11it/s]

...


Epoch 9/20  train_loss=0.007130  lr=5.00e-05


  [ASVspoof-dev] PR-AUC=1.0000  ROC-AUC=0.9997  EER=0.0075  Recall=0.9021  Precision=1.0000  F1=0.9485  Missed=2182
  [ASVspoof-dev] score min=0.000  max=1.000  mean=0.886  std=0.272
  [ASVspoof-dev] Confusion matrix:
[[ 2547     1]
 [ 2182 20114]]
  [Extra-val] PR-AUC=0.9980  ROC-AUC=0.9986  EER=0.0205  Recall=0.9760  Precision=0.9919  F1=0.9839  Missed=3
  [Extra-val] score min=0.002  max=1.000  mean=0.395  std=0.430
  [Extra-val] Confusion matrix:
[[194   1]
 [  3 122]]
  No improvement 4/5  (hmean=0.9476<=best=0.9857)


Epoch 10/20: 100%|██████████| 834/834 [04:34<00:00,  3.04it/s]


Epoch 10/20  train_loss=0.006321  lr=5.00e-05


  [ASVspoof-dev] PR-AUC=1.0000  ROC-AUC=0.9999  EER=0.0027  Recall=0.9621  Precision=1.0000  F1=0.9807  Missed=846
  [ASVspoof-dev] score min=0.001  max=1.000  mean=0.897  std=0.279
  [ASVspoof-dev] Confusion matrix:
[[ 2548     0]
 [  846 21450]]
  [Extra-val] PR-AUC=0.9974  ROC-AUC=0.9982  EER=0.0051  Recall=0.9760  Precision=0.9919  F1=0.9839  Missed=3
  [Extra-val] score min=0.002  max=1.000  mean=0.434  std=0.434
  [Extra-val] Confusion matrix:
[[194   1]
 [  3 122]]
  No improvement 5/5  (hmean=0.9794<=best=0.9857)

Early stopping at epoch 10.  Best hmean=0.9857  Best dev EER=0.0047


In [20]:
def convert(o):
    if isinstance(o, (float, int)): return o
    if hasattr(o, 'item'):   return o.item()
    if hasattr(o, 'tolist'): return o.tolist()
    raise TypeError(f"Not serializable: {type(o)}")
 
with open("/kaggle/working/resnet18_finetune_history.json", "w") as f:
    json.dump(history, f, indent=2, default=convert)
 
print(f"\nTraining complete.")
print(f"Best hmean : {best_hmean:.4f}")
print(f"Best dev EER: {best_dev_eer:.4f}")
 


Training complete.
Best hmean : 0.9857
Best dev EER: 0.0047


In [21]:
best_ckpt = torch.load("/kaggle/working/resnet18_finetuned.pth",
                       map_location=device, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()
 
eval_dataset = UnifiedDataset(
    protocol_path = eval_protocol,
    flac_dir= eval_flac_dir,
    augment= False,
)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)
 
print("\n=== FINAL EVAL — ASVspoof 2019 LA eval set ===")
eval_m = evaluate(model, eval_loader, "ASVspoof-eval")
 
print(f"\nSaved checkpoint: epoch={best_ckpt['epoch']}  "
      f"dev PR-AUC={best_ckpt['pr_auc']:.4f}  EER={best_ckpt['eer']:.4f}")
print(f"\nComparison:")
print(f"  Original:   Recall=0.9201  EER=0.0662  Missed=5101  PR-AUC={ORIGINAL_PR_AUC:.4f}")
print(f"  Finetuned:  Recall={eval_m.get('recall',0):.4f}  "
      f"EER={eval_m.get('eer',1):.4f}  "
      f"Missed={eval_m.get('missed',-1)}  "
      f"PR-AUC={eval_m.get('pr_auc',0):.4f}")
 
with open("/kaggle/working/resnet18_eval_final.json", "w") as f:
    json.dump(eval_m, f, indent=2, default=convert)
print("Eval metrics saved.")


=== FINAL EVAL — ASVspoof 2019 LA eval set ===
  [ASVspoof-eval] PR-AUC=0.9990  ROC-AUC=0.9919  EER=0.0455  Recall=0.9001  Precision=0.9982  F1=0.9466  Missed=6380
  [ASVspoof-eval] score min=0.000  max=1.000  mean=0.824  std=0.312
  [ASVspoof-eval] Confusion matrix:
[[ 7252   103]
 [ 6380 57502]]

Saved checkpoint: epoch=5  dev PR-AUC=1.0000  EER=0.0047

Comparison:
  Original:   Recall=0.9201  EER=0.0662  Missed=5101  PR-AUC=1.0000
  Finetuned:  Recall=0.9001  EER=0.0455  Missed=6380  PR-AUC=0.9990
Eval metrics saved.


In [ ]:
import zipfile

zip_path = "/kaggle/working/resnet18_model.zip"
pth_path = "/kaggle/working/resnet18_finetuned.pth"

with zipfile.ZipFile(zip_path, 'w') as z:
    z.write(pth_path, arcname="resnet18_finetuned.pth")

print("Zipped successfully")